In [2]:
%matplotlib inline

# Importowanie najwazniejszych bibliotek
import numpy as np
import matplotlib.pyplot as plt

# Importowanie najwazniejszych komponentow Sionna RT
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, PathSolver, RadioMapSolver, subcarrier_frequencies, RadioMaterial

In [5]:
scene = load_scene("./OpenStreetMap/kampus_AGH_doppler/Kampus_AGH_B9_doppler.xml", merge_shapes=False)

fc = 28e9
liczba_anten_rx = 16

scene.frequency = fc

# # Tworzenie materiału udającego ludzkie ciało
human_flesh = RadioMaterial("human_flesh", relative_permittivity=45.0, conductivity=1.5, scattering_coefficient=0.2, color=[1.0, 0.4, 0.8])

if "human_flesh" not in scene.radio_materials:
    scene.add(human_flesh)

# Tworzymy chropowaty metal radarowy
car_material = RadioMaterial("radar_target", relative_permittivity=1.0, conductivity=1e7, scattering_coefficient=0.3, color=[0.5, 0.5, 0.5]) 
# car_material = RadioMaterial("radar_target", relative_permittivity=1.0, conductivity=0.0, scattering_coefficient=0.0, color=[0.5, 0.5, 0.5]) 

# Dodajemy go do sceny (zabezpieczenie przed podwójnym dodaniem)
if "radar_target" not in scene.radio_materials:
    scene.add(car_material)

# Szukamy człowieka i samochodu i robimy z nich idealne cele
for name, obj in scene.objects.items():
    if "Samochod" in name:
        obj.radio_material = "radar_target"

# konfiguracja parametrów anteny dla nadajnika
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

# konfiguracja parametrów anteny dla odbiornika
scene.rx_array = PlanarArray(num_rows=1,
                             num_cols=liczba_anten_rx,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="tr38901",
                             polarization="V")

radar_position = [-9.169, 87.045, 17.500]

# Tworzenie nadajnika
tx = Transmitter(name="tx",
                 position=radar_position,
                 display_radius=3)

# dodanie instancji nadajnika do sceny
scene.add(tx)

# utworzenie odbiornika
rx = Receiver(name="rx",
              position=radar_position,
              display_radius=3)

scene.add(rx)

# pobranie referencji do obiektów człowieka i samochodu
czlowiek = scene.get("Czlowiek_001")
samochod = scene.get("Samochod_001")

czlowiek_velocity = [0, -3, 0.0]
samochod_velocity = [-5, -15, 0.0]

scene.get("Czlowiek_001").velocity = czlowiek_velocity
scene.get("Samochod_001").velocity = samochod_velocity

tx.look_at(czlowiek)
rx.look_at(czlowiek)

# Parametry
c = 3e8
wavelength = c / fc

# OFDM
num_ofdm_symbols = 512 
num_subcarriers = 1024
subcarrier_spacing = 120e3

ofdm_symbol_duration = 1 / subcarrier_spacing
delay_resolution = ofdm_symbol_duration / num_subcarriers
doppler_resolution = subcarrier_spacing / num_ofdm_symbols

max_doppler_hz = (num_ofdm_symbols / 2) * doppler_resolution

frequencies = subcarrier_frequencies(num_subcarriers=num_subcarriers, subcarrier_spacing=subcarrier_spacing)

p_solver = PathSolver()

paths = p_solver(scene=scene, max_depth=3, samples_per_src=5_000_000, diffuse_reflection=True, specular_reflection=True)

# generowanie CFR
h = paths.cfr(frequencies=frequencies, sampling_frequency=1 / ofdm_symbol_duration, num_time_steps=num_ofdm_symbols, normalize=False, out_type="numpy")

# Transformacja do dziedziny Dopplera
h = np.squeeze(h)

# Angle-Doppler Map
h_ad_raw = h[:, :, 512].T

h_ad_clean = h_ad_raw - np.mean(h_ad_raw, axis=0, keepdims=True)

window_time_ad = np.hanning(num_ofdm_symbols)[:, None]
window_space_ad = np.hanning(liczba_anten_rx)[None, :]
window_ad = window_time_ad * window_space_ad

h_windowed_ad = h_ad_clean * window_ad

ad_map = np.fft.fftshift(np.fft.fft2(h_windowed_ad))

srodek_y_ad = int(ad_map.shape[0] / 2)

ad_map[srodek_y_ad-2:srodek_y_ad+2, :] = 1e-12

ad_power_db = 20 * np.log10(np.abs(ad_map) + 1e-12)

idx_y, idx_x = np.unravel_index(np.argmax(ad_power_db), ad_power_db.shape)

katy_wektor = np.linspace(-90, 90, ad_power_db.shape[1])
dopplery_wektor = np.linspace(-max_doppler_hz, max_doppler_hz, ad_power_db.shape[0])

wykryty_kat = katy_wektor[idx_x]
wykryta_predkosc_ad = dopplery_wektor[idx_y]

# Range-Doppler Map
h_rd_raw = h[0, :, :]

window_freq_rd = np.hanning(num_subcarriers)[None, :]
h_rd_frequency_windowed = h_rd_raw * window_freq_rd

h_rd_shifted = np.fft.fftshift(h_rd_frequency_windowed, axes=1)
h_delay = np.fft.ifft(h_rd_shifted, axis=1, norm="ortho")

h_delay_clean = h_delay - np.mean(h_delay, axis=0, keepdims=True)

window_time_rd = np.hanning(num_ofdm_symbols)[:, None]
h_delay_windowed = h_delay_clean * window_time_rd
h_range_doppler = np.fft.fft(h_delay_windowed, axis=0, norm="ortho")

rd_map = np.fft.fftshift(h_range_doppler, axes=0)

srodek_y_rd = int(rd_map.shape[0] / 2)
rd_map[srodek_y_rd-2:srodek_y_rd+2, :] = 1e-12

rd_power_db = 20 * np.log10(np.abs(rd_map) + 1e-12)

rd_power_db_half = rd_power_db[:, :int(num_subcarriers / 2)]
idx_y_rd, idx_x_rd = np.unravel_index(np.argmax(rd_power_db_half), rd_power_db_half.shape)

wykryte_opoznienie = idx_x_rd * delay_resolution
wykryty_dystans = (c*wykryte_opoznienie) / 2
wykryta_predkosc_rd = dopplery_wektor[idx_y_rd]

predkosc_ms_ad = (wykryta_predkosc_ad * wavelength) / 2
predkosc_ms_rd = (wykryta_predkosc_rd * wavelength) / 2

predkosc_ad_kmh = predkosc_ms_ad * 3.6
predkosc_rd_kmh = predkosc_ms_rd * 3.6

print(f"Wyniki detekcji:")
print(f"Wykryty kąt: {wykryty_kat:.2f}°")
print(f'Wykryta odległość: {wykryty_dystans:.2f} m')
print(f'Wykryta prędkość (Range-Doppler): {wykryta_predkosc_rd:.2f} Hz, {predkosc_rd_kmh:.2f} km/h')
print(f'Wykryta prędkość (Angle-Doppler): {wykryta_predkosc_ad:.2f} Hz, {predkosc_ad_kmh:.2f} km/h')

print("\nRzeczywistość:")

pos_tx = np.array(tx.position).flatten()
pos_car = np.array(samochod.position).flatten()
pos_human = np.array(czlowiek.position).flatten()
vel_car = np.array(samochod_velocity).flatten()

rzeczywista_odleglosc = float(np.linalg.norm(pos_car - pos_tx))

kat_auta_globalny = np.degrees(np.arctan2(pos_car[1] - pos_tx[1], pos_car[0] - pos_tx[0]))
kat_czlowieka_globalny = np.degrees(np.arctan2(pos_human[1] - pos_tx[1], pos_human[0] - pos_tx[0]))

rzeczywisty_kat_wzgledny = float((kat_auta_globalny - kat_czlowieka_globalny + 180) % 360 - 180)

wektor_kierunkowy = (pos_tx - pos_car) / rzeczywista_odleglosc 
rzeczywista_predkosc_radialna = float(np.dot(vel_car, wektor_kierunkowy) * 3.6)

print(f"Położenie samochodu: {samochod.position}")
print(f"Położenie człowieka (Oś 0° radaru): {czlowiek.position}")
print("-" * 50)
print(f"Rzeczywista prędkość 3D auta: {float(np.linalg.norm(vel_car)) * 3.6:.2f} km/h")
print(f"Rzeczywista odległość do celu: {rzeczywista_odleglosc:.2f} m")
print(f"Rzeczywisty kąt względem radaru: {rzeczywisty_kat_wzgledny:.2f}°")
print(f"Rzeczywista prędkość radialna: {rzeczywista_predkosc_radialna:.2f} km/h")

maksymalny_blad_odleglosci = (c * delay_resolution) / 2
maksymalny_blad_predkosci_ms = (doppler_resolution * wavelength) / 2
maksymalny_blad_predkosci_kmh = maksymalny_blad_predkosci_ms * 3.6
maksymalny_blad_kata = (106.0 / liczba_anten_rx) * 1.5 # Przybliżona formuła na rozdzielczość kątową dla liniowej tablicy antenowej przy użyciu okna Hanninga

print("\nTeoretyczne rozdzielczości detekcji:")
print(f"Rozdzielczość odległości: {maksymalny_blad_odleglosci:.2f} m")
print(f"Rozdzielczość prędkości: {maksymalny_blad_predkosci_kmh:.2f} km/h")
print(f"Rozdzielczość kątowa: {maksymalny_blad_kata:.2f}°")

print("\nAnaliza błędów detekcji:")
blad_odleglosci = abs(wykryty_dystans - rzeczywista_odleglosc)
blad_predkosci_ad = abs(predkosc_ad_kmh - rzeczywista_predkosc_radialna)
blad_predkosci_rd = abs(predkosc_rd_kmh - rzeczywista_predkosc_radialna)
blad_kata = abs(wykryty_kat - rzeczywisty_kat_wzgledny)

print(f"Błąd detekcji odległości: {blad_odleglosci:.2f} m")
print(f"Błąd detekcji prędkości (Angle-Doppler): {blad_predkosci_ad:.2f} km/h")
print(f"Błąd detekcji prędkości (Range-Doppler): {blad_predkosci_rd:.2f} km/h")
print(f"Błąd detekcji kąta: {blad_kata:.2f}°")


Wyniki detekcji:
Wykryty kąt: -6.00°
Wykryta odległość: 29.30 m
Wykryta prędkość (Range-Doppler): -2230.92 Hz, -43.02 km/h
Wykryta prędkość (Angle-Doppler): -2230.92 Hz, -43.02 km/h

Rzeczywistość:
Położenie samochodu: [[-9.63637, 57.3983, 0.7]]
Położenie człowieka (Oś 0° radaru): [[-4.18193, 60.6857, 0.9]]
--------------------------------------------------
Rzeczywista prędkość 3D auta: 56.92 km/h
Rzeczywista odległość do celu: 34.08 m
Rzeczywisty kąt względem radaru: -11.62°
Rzeczywista prędkość radialna: -47.22 km/h

Teoretyczne rozdzielczości detekcji:
Rozdzielczość odległości: 1.22 m
Rozdzielczość prędkości: 4.52 km/h
Rozdzielczość kątowa: 9.94°

Analiza błędów detekcji:
Błąd detekcji odległości: 4.78 m
Błąd detekcji prędkości (Angle-Doppler): 4.20 km/h
Błąd detekcji prędkości (Range-Doppler): 4.20 km/h
Błąd detekcji kąta: 5.62°
